In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 47 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (584 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!which ollama
!ollama --version

/usr/local/bin/ollama


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started!")

Ollama server started!


In [5]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[]}

In [6]:
!ollama pull llama3.2:3b

In [7]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:3b    a80c4f17acd5    2.0 GB    5 seconds ago    


In [8]:
!pip install -q langchain langchain-core langchain-community langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM

print("LangChain imports successful!")

LangChain imports successful!


In [10]:
llm = OllamaLLM(
    model="llama3.2:3b",
    temperature=0
)

print("Ollama LLM created!")

Ollama LLM created!


In [11]:
prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a helpful beginner-friendly AI/ML mentor.

Answer the question in simple and clear language.

Question: {question}

Answer:
"""
)

print("PromptTemplate created!")

PromptTemplate created!


In [12]:
output_parser = StrOutputParser()

print("OutputParser created!")

OutputParser created!


In [13]:
chain = prompt | llm | output_parser

print("LangChain chain created successfully!")

LangChain chain created successfully!


In [14]:
questions = [
    "What is Python?",
    "What is a variable?",
    "What is a function?",
    "What is a list?",
    "What is machine learning?"
]

for i, question in enumerate(questions, 1):

    print("=" * 60)
    print(f"Question {i}: {question}")
    print("=" * 60)

    answer = chain.invoke({
        "question": question
    })

    print(answer)
    print()

Question 1: What is Python?
Python is a programming language that's easy to learn and use. It's like a set of instructions that a computer can understand, and it's used to create all sorts of things like websites, games, and even artificial intelligence models!

Python is known for being simple, flexible, and versatile. It's a popular choice among beginners and experts alike, and it's often used in data science, machine learning, and web development.

Think of Python like a recipe book. You write down a recipe (your code), and the computer follows the steps to make something (like a website or a game). Python makes it easy to write and understand the code, so you can focus on creating something cool!

Question 2: What is a variable?
I'd be happy to explain what a variable is in simple terms!

A variable is a container that holds a value. Think of it like a labeled box where you can store a number, word, or even a picture.

Imagine you have a box with a label that says "Name". You can p

In [15]:
from langchain_core.messages import HumanMessage, AIMessage

print("Memory classes imported!")

Memory classes imported!


In [16]:
conversation_history = []

print("Conversation memory initialized!")

Conversation memory initialized!


In [17]:
memory_prompt = PromptTemplate(
    input_variables=["history", "question"],
    template="""
You are a helpful beginner-friendly AI/ML mentor.

Previous conversation:
{history}

New question:
{question}

Use the previous conversation when it is relevant.

Answer:
"""
)

print("Memory prompt created!")

Memory prompt created!


In [18]:
memory_chain = memory_prompt | llm | output_parser

print("Memory chain created!")

Memory chain created!


In [19]:
def ask_with_memory(question):

    history_text = ""

    for message in conversation_history:

        if isinstance(message, HumanMessage):
            history_text += f"User: {message.content}\n"

        elif isinstance(message, AIMessage):
            history_text += f"AI: {message.content}\n"

    answer = memory_chain.invoke({
        "history": history_text,
        "question": question
    })

    conversation_history.append(
        HumanMessage(content=question)
    )

    conversation_history.append(
        AIMessage(content=answer)
    )

    return answer

In [20]:
turns = [
    "My name is Kusuma.",
    "What is Python?",
    "What can Python be used for?",
    "Which programming language am I learning?",
    "Can you summarize our conversation?"
]

for i, question in enumerate(turns, 1):

    print("=" * 60)
    print(f"TURN {i}")
    print("=" * 60)

    print("User:", question)

    answer = ask_with_memory(question)

    print("AI:", answer)
    print()

TURN 1
User: My name is Kusuma.
AI: Nice to meet you, Kusuma! I'm excited to help you on your AI/ML journey. Unfortunately, this is the beginning of our conversation, so I don't have any previous conversation to draw from. But don't worry, we can start fresh and explore the world of AI/ML together! What's on your mind, Kusuma? Are you looking to learn about a specific topic or get started with a project?

TURN 2
User: What is Python?
AI: Nice to meet you again, Kusuma! I'm glad we can start fresh.

Python is an amazing language that's perfect for beginners like you, especially when it comes to AI and ML. Python is a high-level, interpreted language that's easy to read and write, making it an ideal choice for rapid prototyping and development.

In the context of AI and ML, Python is widely used due to its simplicity, flexibility, and extensive libraries. Many popular AI and ML frameworks, such as TensorFlow, Keras, and scikit-learn, have excellent Python implementations. This means you 

In [21]:
print("===== CONVERSATION HISTORY =====")

for message in conversation_history:

    print(
        type(message).__name__,
        ":",
        message.content
    )

===== CONVERSATION HISTORY =====
HumanMessage : My name is Kusuma.
AIMessage : Nice to meet you, Kusuma! I'm excited to help you on your AI/ML journey. Unfortunately, this is the beginning of our conversation, so I don't have any previous conversation to draw from. But don't worry, we can start fresh and explore the world of AI/ML together! What's on your mind, Kusuma? Are you looking to learn about a specific topic or get started with a project?
HumanMessage : What is Python?
AIMessage : Nice to meet you again, Kusuma! I'm glad we can start fresh.

Python is an amazing language that's perfect for beginners like you, especially when it comes to AI and ML. Python is a high-level, interpreted language that's easy to read and write, making it an ideal choice for rapid prototyping and development.

In the context of AI and ML, Python is widely used due to its simplicity, flexibility, and extensive libraries. Many popular AI and ML frameworks, such as TensorFlow, Keras, and scikit-learn, ha

In [22]:
from langchain_core.tools import tool

@tool
def web_search_stub(query: str) -> str:
    """Simulates a web search."""

    return f"Simulated web search result for: {query}"


@tool
def calculator(expression: str) -> str:
    """Calculates a basic mathematical expression."""

    try:

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception:

        return "Invalid mathematical expression."


print("Two tools created successfully!")

Two tools created successfully!


In [23]:
print("Tool 1:", web_search_stub.name)
print("Tool 2:", calculator.name)

Tool 1: web_search_stub
Tool 2: calculator


In [24]:
def simple_agent(task):

    task_lower = task.lower()

    # Use calculator for mathematical tasks
    if (
        "calculate" in task_lower
        or "+" in task_lower
        or "-" in task_lower
        or "*" in task_lower
        or "/" in task_lower
        or "%" in task_lower
    ):

        expression = task_lower

        for word in [
            "calculate",
            "what is",
            "solve"
        ]:

            expression = expression.replace(
                word,
                ""
            ).strip()

        return calculator.invoke(expression)

    # Otherwise use web search stub
    else:

        return web_search_stub.invoke(task)

In [25]:
agent_tasks = [
    "Calculate 25 * 4",
    "Calculate 100 / 5",
    "Search for LangChain"
]

for i, task in enumerate(agent_tasks, 1):

    print("=" * 60)
    print(f"Agent Task {i}: {task}")
    print("=" * 60)

    result = simple_agent(task)

    print("Result:", result)
    print()

Agent Task 1: Calculate 25 * 4
Result: 100

Agent Task 2: Calculate 100 / 5
Result: 20.0

Agent Task 3: Search for LangChain
Result: Simulated web search result for: Search for LangChain

